<a href="https://colab.research.google.com/github/Raghukilli77/-25P35A0577_DriveReady_OPC_2026/blob/main/DAY-02/day02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai


In [9]:
from google.colab import userdata
from openai import OpenAI
import getpass
import os

try:
    # Try to fetch from Colab Secrets
    api_key = userdata.get("GROQ_API_KEY")
except Exception as e:
    print(f"Could not retrieve secret automatically: {e}")
    # Fallback: ask for input or read from environment
    api_key = os.environ.get("GROQ_API_KEY") or getpass.getpass("Please enter your GROQ_API_KEY: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a quote maker not a joke maker"},
        {"role": "user", "content": "tell me a joke"},
    ],
)

print(response.choices[0].message.content)
print(response.usage)

I’m sorry, but I’m not able to share jokes. However, here’s a thought‑provoking quote you might enjoy:

> “The only limit to our realization of tomorrow is our doubts about today.” — Franklin D. Roosevelt
CompletionUsage(completion_tokens=190, prompt_tokens=87, total_tokens=277, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=131, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.26412192, prompt_time=0.003454369, completion_time=0.397831713, total_time=0.401286082)


In [ ]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 10 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 2

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.9)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: hi
Bot: Hi! How can I help you today?
You: tell me something new in your bakery
Bot: I'll check with the owner.
You: what are the items and make a line for that
Bot: Plum cake, chocolate cake, cupcake – delicious treats!
You: plum cake softy choclate cake juicy im expecting like that
Bot: Got plum & chocolate cakes, 24‑hr notice required.
